# HumMobCov — Main Analysis Notebook

Structured in three sections:
1. **INPUT** — choose the region, load the dataset, inspect configuration
2. **MAIN** — run the full processing pipeline (per-user metric computation)
3. **VISUALIZATION** — produce all plots from the saved per-user results

All paths, parameters and flags are controlled through:
- `src/constants.py` — global constants and directory layout
- `data/config/config_<REGION>.json` — feature flags (which metrics to compute)

---
## 0 · Imports

In [ ]:
import sys
from pathlib import Path

# Make the src package importable regardless of working directory
PROJECT_ROOT = Path().resolve().parent   # HumMobCov/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Central import hub — brings in numpy, pandas, skmob, matplotlib, etc.
# and all project classes / constants
from src import (
    # constants
    PROJECT_ROOT, DIR_SRC, DIR_OUTPUT, DIR_DATA, DIR_CONFIG,
    PERIOD_NAMES, PERIOD_DIVISION, PERIOD_NAMES_TO_DIVISION,
    MIN_POINTS_PER_USER, TIME_THRESHOLD_HOURS, US_BOUNDING_BOX,
    RURALITY_LEVELS, PARTY_NAMES, K_RADIUS_VALUES, LIST_REGIONS,
    # dataset classes
    DataSet_California, DataSet_Massachusets, dataset_info,
    # pipeline
    analyze_from_dataset, analyze_from_s3_progressive, compute_all, get_config,
    # user
    User,
    # plotter
    plotter,
    # storage
    ParquetStore,
    # utilities
    get_already_saved_user_per_period, ifnotexistsmkdir,
)
from src.constants import (
    DIR_MILESTONES_SERVER,
    S3_ENDPOINT_URL, S3_BUCKET, S3_RAW_PREFIX, DIR_SHARD_TEMP,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('Project root:', PROJECT_ROOT)
print('Output dir:  ', DIR_OUTPUT)
print('Data dir:    ', DIR_DATA)
print('S3 endpoint: ', S3_ENDPOINT_URL)
print('S3 bucket:   ', S3_BUCKET)


Project root: /home/aamaduzzi/HumMobCov
Output dir:   /home/aamaduzzi/HumMobCov/output
Data dir:     /home/aamaduzzi/HumMobCov/data


---
## 1 · INPUT

**Edit this section** to choose which region to analyse and to review / override any parameter.

In [2]:
# ─── Choose region ──────────────────────────────────────────────────────────
# Options: "CA"  (California)  or  "MA"  (Massachusetts)
REGION = "CA"

assert REGION in LIST_REGIONS, f"Unknown region '{REGION}'. Choose from {LIST_REGIONS}"

In [3]:
# ─── Optional parameter overrides ───────────────────────────────────────────
# Leave as None to use the defaults from constants.py

OVERRIDE_NP_            = None   # e.g. 30  (minimum stops per user per period)
OVERRIDE_T_THRESHOLD    = None   # e.g. 2   (minimum hours between stops)
OVERRIDE_OUTPUT_DIR     = None   # e.g. Path('/my/custom/output')
OVERRIDE_CONFIG_DIR     = None   # e.g. Path('/my/configs')

In [4]:
# ─── Initialise dataset ─────────────────────────────────────────────────────
if REGION == "CA":
    dataset = DataSet_California()
elif REGION == "MA":
    dataset = DataSet_Massachusets()

# Apply overrides
if OVERRIDE_NP_ is not None:
    dataset.np_ = OVERRIDE_NP_
if OVERRIDE_T_THRESHOLD is not None:
    dataset.t_threshold = OVERRIDE_T_THRESHOLD

print(f"Region:              {dataset.id_}")
print(f"Min points (np_):    {dataset.np_}")
print(f"Time threshold (h):  {dataset.t_threshold}")
print(f"Output directory:    {dataset.dir_output}")
print(f"Number of raw files: {len(dataset.dir_files)}")

Region:              CA
Min points (np_):    20
Time threshold (h):  1
Output directory:    /home/aamaduzzi/HumMobCov/milestones_analysis/CA/dataxuser
Number of raw files: 0


In [5]:
# ─── Preview algorithm-flow configuration ───────────────────────────────────
import json

cfg = get_config(REGION, config_dir=OVERRIDE_CONFIG_DIR)
print("Active feature flags:")
for key, val in cfg.items():
    if not key.startswith('_'):
        flag = '✓' if val else '✗'
        print(f"  {flag}  {key}")

Active feature flags:
  ✓  raw_trajectories
  ✓  is_radius_gyration
  ✗  already_computed_rg
  ✓  is_weekly_radius_gyration
  ✓  is_gonzalez
  ✗  already_computed_gonzalez
  ✓  is_random_entropy
  ✗  already_computed_random_entropy
  ✓  is_uncorrelated_entropy
  ✗  already_computed_uncorrelated_entropy
  ✓  is_real_entropy
  ✗  already_computed_real_entropy
  ✓  is_distance
  ✗  already_computed_distance
  ✓  is_home
  ✗  already_computed_home
  ✓  is_krg
  ✗  already_computed_krg
  ✓  is_St
  ✗  already_computed_St
  ✓  is_fraction_time
  ✗  already_computed_fraction_time
  ✓  is_county_rural
  ✗  already_computed_county_rural
  ✓  is_frequency
  ✗  already_computed_frequency
  ✗  is_week2points
  ✓  save_results


In [6]:
# ─── Preview time periods ────────────────────────────────────────────────────
print("Analysis periods:")
for name, (start, end) in PERIOD_NAMES_TO_DIVISION.items():
    print(f"  {name:25s}  {start.date()}  →  {end.date()}")

Analysis periods:
  15 jan - 15 march          2020-01-15  →  2020-03-15
  15 march - 15 may          2020-03-15  →  2020-05-15
  15 may - sept              2020-05-15  →  2020-09-30


---
## 2 · MAIN

Runs the full per-user computation pipeline, selecting the appropriate execution mode automatically:

| Mode | Condition | Action |
|---|---|---|
| **A — local raw** | Raw parquet shards present on local filesystem | `analyze_from_dataset()` — standard batch processing |
| **B — S3 progressive** | `raw_trajectories=true` in config but files not local | `analyze_from_s3_progressive()` — download one shard at a time, compute, delete |
| **C-CA — legacy migration** | CA only; per-user CSV.gz files in `dataxuser/` | `store.migrate_all_periods()` — one-time migration to parquet |
| **C-MA — legacy migration** | MA only; per-metric shard dirs in `milestones_analysis/MA/` | `store.migrate_all_periods_MA()` — one-time migration to parquet |
| **Done** | All periods already in parquet store | Skip — go to Section 3 |

**The pipeline is resume-safe at two levels:**
- *Shard level* (S3 mode): `shard_checkpoint_np_{np_}_t_{t}.json` records which raw files have been fully processed. Interrupted runs restart from the next unprocessed shard.
- *User level*: users already present in the parquet store are detected in O(1) via footer metadata and skipped unconditionally.

**S3 configuration** (override via environment variables if needed):
```
S3_ENDPOINT_URL  https://s3.atlas.fbk.eu   (env: S3_ENDPOINT_URL)
S3_BUCKET        chub-datalake              (env: S3_BUCKET)
S3_PREFIX_CA     shared/cuebiq/MOBS/urban_rural_flow_stops_cali_urban_rural_v3  (env: S3_PREFIX_CA)
S3_PREFIX_MA     shared/cuebiq/MOBS/20220330_stops_hq_users_MA                  (env: S3_PREFIX_MA)
SHARD_TEMP_DIR   HumMobCov/.shard_tmp       (env: SHARD_TEMP_DIR)
```


In [ ]:
# ─── Initialise parquet store ────────────────────────────────────────────────
store = ParquetStore(
    base_dir     = DIR_MILESTONES_SERVER / REGION,
    np_          = dataset.np_,
    t_threshold  = dataset.t_threshold,
)

# ─── Check what is already computed ──────────────────────────────────────────
periods_done = [
    p for p in PERIOD_NAMES
    if len(store.get_computed_users(p, "all_scalars")) > 0
]
periods_todo = [p for p in PERIOD_NAMES if p not in periods_done]

print(f"Periods already in store : {periods_done or 'none'}")
print(f"Periods still to compute : {periods_todo or 'none'}")

if not periods_todo:
    print("\nParquet store already populated for all periods. Nothing to do.")

else:
    # ── CASE A: local raw parquet files ──────────────────────────────────────
    local_raw_files = [f for f in dataset.dir_files if Path(f).exists()]

    if cfg.get("raw_trajectories") and local_raw_files:
        print(f"\nMODE A — local raw data ({len(local_raw_files)} shards found).")
        analyze_from_dataset(
            dataset,
            region     = REGION,
            config_dir = OVERRIDE_CONFIG_DIR,
            output_dir = OVERRIDE_OUTPUT_DIR,
            store      = store,
            batch_size = 500,
        )
        print("Computation complete. Consolidating shards…")
        for p in dataset.period_names:
            store.consolidate_all(p)
        print("Done.")

    # ── CASE B: S3 progressive (download one shard → compute → delete) ───────
    elif cfg.get("raw_trajectories"):
        print(f"\nMODE B — S3 progressive download.")
        print(f"  Endpoint : {S3_ENDPOINT_URL}")
        print(f"  Bucket   : {S3_BUCKET}")
        print(f"  Prefix   : {S3_RAW_PREFIX[REGION]}")
        print(f"  Temp dir : {DIR_SHARD_TEMP}")
        print(f"  Periods to compute: {periods_todo}\n")
        analyze_from_s3_progressive(
            dataset,
            region       = REGION,
            cfg          = cfg,
            store        = store,
            endpoint_url = S3_ENDPOINT_URL,
            bucket       = S3_BUCKET,
            s3_prefix    = S3_RAW_PREFIX[REGION],
            temp_dir     = DIR_SHARD_TEMP,
            batch_size   = 500,
            consolidate_every = 1,   # consolidate after every shard
        )

    # ── CASE C-CA: migrate legacy dataxuser per-user CSV.gz files ────────────
    elif REGION == "CA":
        legacy_dir = DIR_MILESTONES_SERVER / REGION / "dataxuser"
        if legacy_dir.exists() and any(legacy_dir.iterdir()):
            print(
                f"\nMODE C-CA — migrating legacy dataxuser/ CSV.gz files.\n"
                f"  Periods missing: {periods_todo}\n"
                f"  (Already-migrated users are skipped — safe to re-run.)"
            )
            store.migrate_all_periods(
                legacy_dir   = legacy_dir,
                period_names = periods_todo,
                np_          = dataset.np_,
                t            = dataset.t_threshold,
                batch_size   = 5000,
                consolidate  = True,
            )
        else:
            print(
                f"\nCA: no dataxuser/ directory found at {legacy_dir}.\n"
                f"Set raw_trajectories=true in config_CA.json to compute from S3."
            )

    # ── CASE C-MA: migrate legacy per-metric shard directories ───────────────
    elif REGION == "MA":
        ma_legacy_base = DIR_MILESTONES_SERVER / "MA"
        print(
            f"\nMODE C-MA — migrating legacy metric-folder shard files.\n"
            f"  Periods missing: {periods_todo}\n"
            f"  np_={dataset.np_}, t={dataset.t_threshold}\n"
            f"  (Already-migrated users are skipped — safe to re-run.)"
        )
        store.migrate_all_periods_MA(
            ma_legacy_base = ma_legacy_base,
            period_names   = periods_todo,
            np_            = dataset.np_,
            t              = dataset.t_threshold,
            batch_size     = 2000,
            consolidate    = True,
        )

    else:
        print(f"Unknown region {REGION!r}. No action taken.")


CA: migrating legacy per-user CSV.gz files from dataxuser/ …
  Periods with no data yet: ['15 march - 15 may', '15 may - sept']
  (Already-migrated users are skipped — safe to re-run.)

Migrating period: '15 march - 15 may' …
  Period '15 march - 15 may': 0 users migrated to parquet store.
  Consolidating shards for '15 march - 15 may' …

Migrating period: '15 may - sept' …
  Period '15 may - sept': 0 users migrated to parquet store.
  Consolidating shards for '15 may - sept' …

Migration complete.


In [9]:
# ─── Pipeline summary ────────────────────────────────────────────────────────
# Show how many users are available per period and metric kind.
# Reads only parquet footer metadata (no data loaded).

print("Users available in parquet store:")
for period in PERIOD_NAMES:
    scalars = store.get_computed_users(period, "all_scalars")
    gonzalez = store.get_computed_users_long(period, "gonzalez")
    st = store.get_computed_users(period, "S")
    wrg = store.get_computed_users(period, "weekly_rg")
    freq = store.get_computed_users_long(period, "frequency")
    print(
        f"  {period:25s}  "
        f"scalars={len(scalars):>7,}  "
        f"gonzalez={len(gonzalez):>7,}  "
        f"S(t)={len(st):>7,}  "
        f"weekly_rg={len(wrg):>7,}  "
        f"freq={len(freq):>7,}"
    )


Users available in parquet store:
  15 jan - 15 march          scalars=1,438,004  gonzalez=      0  S(t)=      0  weekly_rg=      0  freq=      0
  15 march - 15 may          scalars=      0  gonzalez=      0  S(t)=      0  weekly_rg=      0  freq=      0
  15 may - sept              scalars=      0  gonzalez=      0  S(t)=      0  weekly_rg=      0  freq=      0


---
## 3 · VISUALIZATION

All plots are produced from the already-saved per-user files.
Run this section independently of Section 2 once results are on disk.

In [10]:
# ─── Initialise plotter ──────────────────────────────────────────────────────
# Pass `store=store` to use the fast parquet backend.
# Omit it (or pass None) to fall back to the legacy per-file mode.

plt_obj = plotter(
    np_            = dataset.np_,
    period_division= dataset.period_division,
    period_names   = dataset.period_names,
    t_threshold    = dataset.t_threshold,
    region         = REGION,
    county2party   = dataset.county2party,
    df_rurality    = dataset.df_rurality,
    output_dir     = OVERRIDE_OUTPUT_DIR,
    store          = store,          # ← parquet-store mode
)

# Show user counts per period (from parquet store metadata)
print("Users available per period (from parquet store):")
for period in plt_obj.period_names:
    n = len(store.get_computed_users(period, "all_scalars"))
    print(f"  {period:25s}  {n:>8,} users")


Users available per period (from parquet store):
  15 jan - 15 march          1,438,004 users
  15 march - 15 may                 0 users
  15 may - sept                     0 users


### 3.1 Radius of Gyration

In [ ]:
plt_obj.plot_rg()

In [ ]:
plt_obj.plot_rg_party_per_period()

In [ ]:
plt_obj.plot_rg_rurality_per_period()

### 3.2 Weekly Radius of Gyration

In [ ]:
plt_obj.plot_weekly_rg()

In [ ]:
plt_obj.plot_rg_rurality_weekly()

In [ ]:
plt_obj.plot_rg_party_weekly()

### 3.3 k-Radius of Gyration

In [ ]:
plt_obj.plot_krg()

### 3.4 Distance

In [ ]:
plt_obj.plot_distance()

### 3.5 Entropy

In [ ]:
plt_obj.plot_entropy()

### 3.6 Exploration Curve S(t)

In [ ]:
plt_obj.plot_St()

### 3.7 Location Frequency

In [ ]:
plt_obj.plot_frequency()

### 3.8 Gonzalez Trajectory Shape

In [ ]:
plt_obj.plot_gonzalez()

In [ ]:
plt_obj.plot_sigmaxy()